# Collimated Comparison

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

In [ ]:
import diffinytrace as dit
from diffinytrace.source import CollimatedMonochromatic,CollimatedGaussianBeam
from diffinytrace.nonimaging.smoothing import GaussianSmootherSquare
import torch
import matplotlib.pyplot as plt

def run_optimiaztion(sigma,use_desired_irradiance_smoothing,return_system=False):
    in_aperture = 4.0
    in_aperture_lens = 5.0

    desired_width_square = 4.0

    out_aperture = 8.0
    source_wl = 0.589
    source_gaussian_constant = 0.035

    light_transform = dit.transforms.Identity()
    source = CollimatedGaussianBeam(light_transform,in_aperture,source_wl,source_gaussian_constant)

    lens_mat = dit.materials["NBK7"]
    env_mat = dit.materials["NONE"]

    lens1_thickness = 2.
    lens1_surf1 = dit.Bspline(in_aperture_lens,[4,4],[11,11])#dit.Legendre(in_aperture_lens,20)#
    lens1_surf2 = dit.Plane()

    det_surf = dit.Plane()

    lens1_transform = dit.transforms.Distance(5.0,parent_transform=source)
    lens1_transform.distance.requires_grad = False

    lens1 = dit.Lens(lens1_transform,lens1_thickness,lens1_surf1,lens1_surf2,lens_mat,in_aperture_lens,is_square=False)


    det_transform = dit.transforms.Distance(10.,parent_transform=lens1)
    det_transform.distance.requires_grad = False
    detector = dit.Detector(det_transform,det_surf,out_aperture)

    system = dit.SequentialOpticalSystem({"source":source,"lens1":lens1,"detector":detector},env_mat)
    system.to("cuda:0")
    sequence = ["source","lens1","detector"]

    N = 2**15

    def desired_irradiance_func(y):
        out = (torch.abs(y[:,0]) < desired_width_square).float() * (torch.abs(y[:,1]) < desired_width_square).float()
        return out

    smoother = GaussianSmootherSquare(out_aperture,
                                    num_conv_points=128,
                                    sigma=sigma,
                                    num_integration_points_desired=[701,701],
                                    desired_irradiance_func=desired_irradiance_func)
    out = smoother.desired_none_smooth_irradiance_opti
    #dit.plotting.quantity2D.plot(out,"Desired Irradiance [W/mm²]",out_aperture)
    #plt.savefig("collimated_desired_irr.png")
    """
    
    x,weights = source.sample(N,method="sobol")
    x = x.to("cuda:0")
    weights = weights.to("cuda:0")

    ray_multi = source.get_flux(x)*weights
    O2, D2, wl, n_func_enviroment, meta_data = system(x,sequence)
    local_pos = detector.to_local_pos(O2)[:,[0,1]]  
    out = smoother.get_none_smooth_irradiance(local_pos,ray_multi)

    dit.plotting.quantity2D.plot(out,"Irradiance Initial",out_aperture)

    x,weights = source.sample(10)
    x = x.to("cuda:0")
    weights = weights.to("cuda:0")
    O2, D2, wl, n_func_enviroment, meta_data = system(x,sequence)
    dit.plotting.system2D.plot(system,meta_data)
    """

    convergence_list = []
    def merit_func():    
        x,weights = source.sample(N,"sobol_pow2")
        x = x.to("cuda:0")
        weights = weights.to("cuda:0")
        ray_multi = source.get_flux(x)*weights
        O2, D2, wl, n_func_enviroment, meta_data = system(x,sequence)
        local_pos = detector.to_local_pos(O2)[:,[0,1]]
        out = smoother.get_merit_function_value(local_pos,
                                                ray_multi,
                                                wl,
                                                save_last_eval=True,
                                                use_desired_irradiance_smoothing=use_desired_irradiance_smoothing)
        return out


    def convergence_callback_func(smoother:dit.nonimaging.smoothing.Smoother,convergence_list:list):
        def return_func():
            convergence_list.append(smoother.last_eval_merit_val)
            
        return return_func 

    result = dit.optimize.minimize(merit_func,system.parameters(),callback=convergence_callback_func(smoother,convergence_list),method="L-BFGS-B",save_history=True,call_before_minimize=True)
    result["sigma"] = sigma
    result["use_desired_irradiance_smoothing"] = use_desired_irradiance_smoothing
    result["convergence_list"] = convergence_list
    
    x,weights = source.sample(N,method="sobol")
    x = x.to("cuda:0")
    weights = weights.to("cuda:0")
    ray_multi = source.get_flux(x)*weights
    O2, D2, wl, n_func_enviroment, meta_data = system(x,sequence)
    local_pos = detector.to_local_pos(O2)[:,[0,1]]  
    none_smooth_irr = smoother.get_none_smooth_irradiance(local_pos,ray_multi)
    smooth_irr = smoother.get_smooth_irradiance(local_pos,ray_multi)

    result["smooth_irr"] = smooth_irr.cpu().detach()
    result["none_smooth_irr"] = none_smooth_irr.cpu().detach()
    if return_system:
        return result,system
    
    return result


In [ ]:
import torch

sigmas = torch.linspace(0.05,1.,20) 
def run_all(use_desired_irradiance_smoothing):
    all_results = []
    for sigma in sigmas:
        all_results.append(run_optimiaztion(sigma,use_desired_irradiance_smoothing))
    return all_results

baseline_results = run_all(use_desired_irradiance_smoothing=False)
our_results = run_all(use_desired_irradiance_smoothing=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
ax = plt.gca()
ax.minorticks_on()  # Enable minor ticks
ax.grid(True, which='minor', linestyle=':', linewidth=0.5)  # Minor grid lines (finer)
ax.set_xlabel("#Iterations")
ax.set_ylabel("Error")

for result in baseline_results:
    fun_vals = result["history"]["fun_vals"]
    sigma = result["sigma"]
    plt.plot(np.arange(len(fun_vals)),fun_vals,label=f"$\\sigma={sigma}$")
plt.legend()


In [ ]:

baseline_conv = [result["convergence_list"][-1] for result in baseline_results]
our_conv = [result["convergence_list"][-1] for result in our_results]
sigmas = [result["sigma"] for result in our_results]
ax = plt.gca()
ax.grid(True, which='major', linestyle=':', linewidth=0.5)  # Minor grid lines (finer)
ax.set_xlabel("$\\sigma$ [mm]")
ax.set_ylabel("Error")
plt.plot(sigmas,baseline_conv,label="Partially Smoothed")
plt.plot(sigmas,our_conv,label="Ours")
plt.title("Relationship Between Error and the Kernel Width")
plt.legend()


idx = torch.arange(len(baseline_results)//4)*5
idx = [0,5,10,15,19]


In [ ]:
xbaseline_results = [baseline_results[i] for i in idx]
xour_results = [our_results[i]for i in idx]
xbaseline_results
data_grid = [[]]*4
#result["smooth_irr"] = smooth_irr.cpu().detach()
#result["none_smooth_irr"] = none_smooth_irr.cpu().detach()
    
data_grid[0] = [result["smooth_irr"] for result in xbaseline_results]
data_grid[1] = [result["smooth_irr"] for result in xour_results]
data_grid[2] = [result["none_smooth_irr"] for result in xbaseline_results]
data_grid[3] = [result["none_smooth_irr"] for result in xour_results]


In [ ]:
from PIL import Image, ImageDraw, ImageFont
from image_grid_maker import image_from_grid
out_aperture = 8.0
    
rows_extent = [[-out_aperture,out_aperture,-out_aperture,out_aperture]]*4
rows_vidx = ["smooth","smooth","non","non"]
rows_cmap = ["jet"]*4
cbar_titles = ["[W/mm²]"]*4
columns_title = [f'σ={result["sigma"]} mm' for result in baseline_results]
columns_title = [columns_title[i] for i in idx]
rows_title = ["(Partially Smoothed)\nSmooth Irr.","(Ours)\nSmooth Irr.","(Partially Smoothed)\nIrr. RC","(Ours)\nIrr. RC"]

kwargs = dict(
        image_grid=data_grid,
        rows_extent=rows_extent,
        rows_vidx=rows_vidx,
        rows_cmap=rows_cmap,
        rows_title=rows_title,
        cbar_titles=cbar_titles,
        columns_title=columns_title,
)
out = image_from_grid(
    **kwargs,
    max_num_column=len(columns_title),
    font_size_PIL=40,
    cbar_labelsize=20,
    cbar_title_fontsize=20,
    column_title_ratio=0.3
    )
out = out[0]
    


# Load an image from the file path
image = Image.open(out)
image.save("comparison_collimated.png")

result,system = run_optimiaztion(sigmas[0],use_desired_irradiance_smoothing=True,return_system=True)



In [ ]:
out_aperture = 8.0

import numpy as np
import plotly.graph_objects as go

system.cpu()
source = system.modules_dict["source"]
lens1 = system.modules_dict["lens1"]
detector = system.modules_dict["detector"]

data = []

resolution = 32
data += dit.plotting.system3D._plot_surface_recursively(lens1,"",resolution)
data += dit.plotting.system3D._plot_surface_recursively(source,"",resolution)

#%%
num_bins = 512
x = np.linspace(-out_aperture, out_aperture, num_bins)  # Width
y = np.linspace(-out_aperture, out_aperture, num_bins)  # Height
z = torch.zeros((num_bins, num_bins))

tmp = np.meshgrid(x,y)
tmp = torch.tensor([tmp])[0].reshape(2,-1).T

irr_source = source.get_flux(tmp)
irr_source[torch.linalg.norm(tmp,dim=1)>out_aperture]=torch.nan
irr_source = irr_source.reshape(num_bins,num_bins)

dit.plotting.quantity2D.plot(irr_source,title="Radiant Exitance [W/mm²]",x_range=[-out_aperture,out_aperture],cmap="hot")


In [ ]:
xr,_ = source.sample(15)
sequence = ["source","lens1","detector"]

O,D,wave_len,_,RayPaths = system(xr,sequence)

if not RayPaths is None:
    if isinstance(RayPaths,dict):
        rays = RayPaths["ray_paths"]
        
show_grid=True
xlabel="x [mm]"
ylabel="y [mm]"
zlabel="z [mm]"
xticks=None
yticks=None
zticks=None
axislabel_font_size=10
tick_font_size=10
ray_color="#9673A6"
ray_linewidth=3.

data += dit.plotting.system3D.ray_paths(rays,ray_color,ray_linewidth)
layout = dit.plotting.system3D.get_optical_system_layout(False,xlabel,ylabel,zlabel,xticks,yticks,zticks,axislabel_font_size,tick_font_size)


import plotly.graph_objects as go
none_smooth_irr = result["none_smooth_irr"]

num_bins = none_smooth_irr.shape[0]

out_aperture = 8.0

x = np.linspace(-out_aperture, out_aperture, num_bins)  # Width
y = np.linspace(-out_aperture, out_aperture, num_bins)  # Height
z = torch.ones((num_bins, num_bins))*detector.get_transformation_matrix()[2,3]        # Flat surface
z = z.detach()
gosurface2 = go.Surface(
        x=x,
        y=y,
        z=z,
        surfacecolor=none_smooth_irr,
        #cmin=0,
        #cmax=1.,
        colorbar=dict(
        title=dict(text='[W/mm²]', font=dict(size=16)),  # Increase title font size
        tickfont=dict(size=14)  # Increase tick font size
        ),
        colorscale="jet",
        showscale=True,
        showlegend=False
)

data += [gosurface2]
fig = go.Figure(data=data,layout=layout)
fig.write_html("collimated2.html")
fig.show()
